# Fundamentals 00.1 - Runtime Scheduler API

Objetivo: observar como toolkit.scheduler protege ejecuciones reales sin cambiar la ruta publica runtime -> system -> agent -> RunResult.

## Parametros de la demostracion

| Parametro | Valor | Proposito |
|---|---:|---|
| timeout_s | 30 | Limitar la duracion de una ejecucion normal. |
| max_retries | 1 | Recuperar una falla transitoria controlada. |
| max_tool_calls | 2 | Acotar el trabajo del agente. |
| max_turns | 3 | Evitar ciclos no intencionales. |

## 1) Runtime declarativo

El scheduler es politica operativa. El runtime selecciona el provider y el System registra componentes; ninguno ejecuta trabajo durante su construccion.

In [ ]:
import time

import agentic_systems as toolkit

scheduler = toolkit.scheduler(
    timeout_s=30,
    max_retries=1,
    max_tool_calls=2,
    max_turns=3,
    max_concurrency=1,
    backoff_s=0.0,
)
runtime = toolkit.runtime(
    provider="python-runtime",
    scheduler=scheduler,
    metadata={"tutorial": "runtime_scheduler_api"},
)
system = toolkit.system(runtime=runtime)

toolkit.show_json(
    {"scheduler": scheduler, "runtime": runtime, "system": system.inspect()},
    title="Configuracion observable",
)

## 2) Ejecucion normal

La Tool consulta el contrato publico real. El usuario puede cambiar symbol sin modificar la implementacion ni recibir una respuesta precocinada.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    """Describe si un simbolo pertenece a la API publica instalada."""
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "public_api_size": len(toolkit.__all__),
    }


agent = system.agent(
    name="scheduler_api_agent",
    instructions="Inspecciona el simbolo solicitado y conserva la evidencia de la Tool.",
    tools=[inspect_public_api],
    runtime=runtime,
)
result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "scheduler"}},
    mode="eval",
)
assert isinstance(result, toolkit.RunResult)

toolkit.human_result(result, title="RunResult normal")
toolkit.show_json(
    {
        "output": toolkit.run_result_output(result),
        "scheduler_execution": result.meta.get("scheduler_execution"),
        "usage_scheduler": result.usage.get("scheduler"),
    },
    title="Envelope observable",
)

## 3) Retry controlado

La primera llamada falla de forma transitoria. max_retries=1 permite que el mismo contrato se recupere y deja la evidencia en el RunResult.

In [ ]:
attempts = {"count": 0}


@toolkit.tool
def unstable_lookup(symbol: str) -> dict:
    """Simula una dependencia transitoria y luego consulta PUBLIC_API."""
    attempts["count"] += 1
    if attempts["count"] == 1:
        raise RuntimeError("transient dependency failure")
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "attempt": attempts["count"],
    }


retry_runtime = toolkit.runtime(
    provider="python-runtime",
    scheduler=toolkit.scheduler(
        timeout_s=30,
        max_retries=1,
        max_tool_calls=2,
        max_turns=3,
        backoff_s=0.0,
    ),
)
retry_system = toolkit.system(runtime=retry_runtime)
retry_agent = retry_system.agent(
    name="retry_api_agent",
    instructions="Reintenta la consulta y conserva solo evidencia observada.",
    tools=[unstable_lookup],
    runtime=retry_runtime,
)
retry_result = retry_agent.run(
    {"tool": "unstable_lookup", "input": {"symbol": "RunResult"}},
    mode="eval",
)

toolkit.human_result(retry_result, title="RunResult recuperado")
toolkit.show_json(
    {
        "attempts": attempts["count"],
        "scheduler_execution": retry_result.meta.get("scheduler_execution"),
    },
    title="Evidencia del retry",
)

## 4) Timeout observable

Una Tool deliberadamente lenta excede el limite. El fallo se expresa en el mismo envelope RunResult; el notebook no fabrica ni muta el resultado.

In [ ]:
@toolkit.tool
def delayed_lookup(symbol: str) -> dict:
    """Demora la consulta para demostrar el timeout del scheduler."""
    time.sleep(0.2)
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}


timeout_runtime = toolkit.runtime(
    provider="python-runtime",
    scheduler=toolkit.scheduler(
        timeout_s=0.05,
        max_retries=0,
        max_tool_calls=1,
        max_turns=1,
    ),
)
timeout_system = toolkit.system(runtime=timeout_runtime)
timeout_agent = timeout_system.agent(
    name="timeout_api_agent",
    instructions="Ejecuta la consulta dentro de los limites declarados.",
    tools=[delayed_lookup],
    runtime=timeout_runtime,
)
timeout_result = timeout_agent.run(
    {"tool": "delayed_lookup", "input": {"symbol": "scheduler"}},
    mode="eval",
)

toolkit.human_result(timeout_result, title="RunResult con timeout")
toolkit.show_json(
    {
        "ok": timeout_result.ok,
        "errors": timeout_result.errors,
        "scheduler_execution": timeout_result.meta.get("scheduler_execution"),
    },
    title="Evidencia del timeout",
)

## Lo importante

- scheduler declara limites y reintentos; no sustituye al runtime.
- runtime conserva la seleccion del provider.
- system.agent materializa la ruta publica sin acoplarse a SDKs.
- exito, retry y timeout terminan en el mismo contrato RunResult.

## Coverage API de este notebook

La cobertura se declara como datos para que el gate pueda compararla contra toolkit.__all__.

In [ ]:
api_coverage = [
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.system",
    "toolkit.tool",
    "system.agent",
    "agent.run",
    "toolkit.RunResult",
    "toolkit.run_result_output",
    "toolkit.human_result",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Scheduler API coverage")